In [1]:
!python -m pip install --upgrade pip
!python -m pip install --upgrade pandas openpyxl openai pycelonis_llm pymupdf ipykernel
!python -m pip install --upgrade brotli brotlicffi

Looking in indexes: https://pypi.celonis.cloud, https://pypi.org/simple
  Using cached pip-26.3-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.3-py3-none-any.whl (2.0 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Looking in indexes: https://pypi.celonis.cloud, https://pypi.org/simple
  Using cached pandas-3.1.1-cp311-cp311-manylinux_2_24_aarch64.manylinux_2_28_aarch64.whl.metadata (16 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached openai-3.49.0-py3-none-any.whl.metadata (45 kB)
  Using cached pycelonis_llm-0.3.0-py3-none-any.whl (11 kB)
  Using cached pymupdf-2.1-cp311-abi3-manylinux_2_28_aarch64.whl.metadata (26 kB)
  Using cached ipykernel-7.4.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_aarch64.manylinux_2_28_aarch64.whl.metadata (6.6 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata 

This cell checks the new documents which were submitted and their current status signature.

In [2]:
import pandas as pd

# Read the document status file, keeping every column as a string
doc_status_df = pd.read_excel("DocumentStatus.xlsx", dtype=str)

# Keep only the documents whose Status is null
null_status_df = doc_status_df.loc[
    doc_status_df["Status"].isna(), ["KYCAssessmentID", "DocumentNo", "CreationDate", "LastUpdated"]
].reset_index(drop=True)

print(f"Documents with null status: {len(null_status_df)} of {len(doc_status_df)}")
null_status_df

Documents with null status: 3 of 3


,KYCAssessmentID,DocumentNo,CreationDate,LastUpdated
0,921632,001,2026-11-10 00:00:00,NaN
1,927015,002,2026-11-10 00:00:00,NaN
2,954350,003,2026-11-10 00:00:00,NaN


This cell loads the PDF files which do not have a Status yet.

In [3]:
import base64
from pathlib import Path

DOCUMENTS_DIR = Path("Documents")

def read_pdf(document_no):
    """Return the raw bytes of Documents/<DocumentNo>.pdf, or None if it is missing."""
    pdf_path = DOCUMENTS_DIR / f"{document_no}.pdf"
    return pdf_path.read_bytes() if pdf_path.exists() else None

# Read the PDF for each document with a null status
null_status_df["PdfBytes"] = null_status_df["DocumentNo"].apply(read_pdf)
null_status_df["PdfBase64"] = null_status_df["PdfBytes"].apply(
    lambda b: base64.standard_b64encode(b).decode("utf-8") if b is not None else None
)

missing = null_status_df.loc[null_status_df["PdfBytes"].isna(), "DocumentNo"].tolist()
print(f"PDFs loaded: {len(null_status_df) - len(missing)} of {len(null_status_df)}")
if missing:
    print(f"Missing PDFs for DocumentNo: {missing}")
null_status_df[["KYCAssessmentID", "DocumentNo"]].assign(
    PdfSizeKB=null_status_df["PdfBytes"].apply(lambda b: round(len(b) / 1024, 1) if b else None)
)

PDFs loaded: 3 of 3


,KYCAssessmentID,DocumentNo,PdfSizeKB
0,921632,001,1117.9
1,927015,002,861.7
2,954350,003,195.1


This cell checks for digital signatures. If a digital signature is detected, then it updates the status to "Digital Signature Detected", and changes the LastUpdated status to the time of the runtime.

In [4]:
import re
from datetime import datetime

# A signed PDF contains a signature dictionary with a /ByteRange entry that lists the
# signed byte ranges. Empty (unsigned) signature fields have no /ByteRange.
BYTE_RANGE_PATTERN = re.compile(rb"/ByteRange\s*\[")

def has_digital_signature(pdf_bytes):
    """Return True if the PDF contains at least one applied digital signature."""
    return pdf_bytes is not None and BYTE_RANGE_PATTERN.search(pdf_bytes) is not None

null_status_df["HasDigitalSignature"] = null_status_df["PdfBytes"].apply(has_digital_signature)

# Timestamp for every status changed in this run, as a string to match dtype=str
last_updated = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Update the status of digitally signed documents
signed_mask = null_status_df["HasDigitalSignature"]
null_status_df["Status"] = None
null_status_df["LastUpdated"] = None
null_status_df.loc[signed_mask, "Status"] = "Digital Signature Detected"
null_status_df.loc[signed_mask, "LastUpdated"] = last_updated

# Carry the new status back to the full document status dataframe
signed_document_nos = null_status_df.loc[signed_mask, "DocumentNo"]
doc_status_df.loc[
    doc_status_df["DocumentNo"].isin(signed_document_nos), ["Status", "LastUpdated"]
] = ["Digital Signature Detected", last_updated]

print(f"Digital signatures detected: {signed_mask.sum()} of {len(null_status_df)}")
null_status_df[["KYCAssessmentID", "DocumentNo", "HasDigitalSignature", "Status", "LastUpdated"]]

Digital signatures detected: 0 of 3


,KYCAssessmentID,DocumentNo,HasDigitalSignature,Status,LastUpdated
0,921632,001,False,None,None
1,927015,002,False,None,None
2,954350,003,False,None,None


This section of code checks for the remaining documents which do not have a digital signature to see if there is a manual or no signature and updates the status of the document accordingly.

In [5]:
from pycelonis_llm.integrations.openai import CelonisOpenAI

# No API key is needed: CelonisOpenAI takes its credentials from the Celonis environment
client = CelonisOpenAI()
print("Celonis OpenAI client ready")

[2026-11-10 17:08:11,175] INFO: No `base_url` given. Using environment variable 'CELONIS_URL'


[2026-11-10 17:08:11,815] WARNING: KeyType is not set. Defaulted to 'APP_KEY'.


[2026-11-10 17:08:11,863] INFO: Initial connect successful! PyCelonis Version: 2.18.0
Celonis OpenAI client ready


In [6]:
import base64
import json
from datetime import datetime

import fitz  # PyMuPDF, renders PDF pages to images
import openai

# `client` is created in the Celonis cell above
MODEL = "azure-openai-gpt-5"  # Celonis model ID for GPT-5.6 Luna
SIGNATURE_PROMPT = "Does this file contain a signature? Check at the bottom of the image for a handwritten signature. Return with options Manual Signature and No Signature"
SIGNATURE_OPTIONS = ["Manual Signature", "No Signature"]
RENDER_DPI = 250

def pdf_to_image_parts(pdf_bytes, dpi=RENDER_DPI):
    """Render every page of the PDF to a PNG and return them as chat image_url content parts."""
    parts = []
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        for page in doc:
            png = page.get_pixmap(dpi=dpi).tobytes("png")
            parts.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{base64.b64encode(png).decode('utf-8')}"},
            })
    return parts

def classify_signature(pdf_bytes):
    """Send the pages of one PDF to GPT-5.6 Luna and return "Manual Signature" or "No Signature", or None on failure."""
    completion = client.chat.completions.create(
        model=MODEL,
        # The installed brotli library is too old for httpx2, so ask for gzip instead of br
        extra_headers={"Accept-Encoding": "gzip, deflate"},
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "signature_result",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {"signature": {"type": "string", "enum": SIGNATURE_OPTIONS}},
                    "required": ["signature"],
                    "additionalProperties": False,
                },
            },
        },
        messages=[
            {"role": "system", "content": "You check documents for signatures."},
            {
                "role": "user",
                "content": pdf_to_image_parts(pdf_bytes) + [{"type": "text", "text": SIGNATURE_PROMPT}],
            },
        ],
    )
    choice = completion.choices[0]
    # The Celonis gateway leaves finish_reason as None on normal completions, so accept it.
    # Anything else (e.g. "length", "content_filter") means the answer is cut off or blocked.
    if choice.finish_reason not in (None, "stop"):
        print(f"Unexpected finish reason '{choice.finish_reason}' (completion {completion.id})")
        return None
    # Success is decided by the answer itself: only a missing, refused or unparsable answer fails
    if choice.message.refusal or not choice.message.content:
        print(f"No answer returned, refusal: {choice.message.refusal} (completion {completion.id})")
        return None
    try:
        signature = json.loads(choice.message.content)["signature"]
    except (json.JSONDecodeError, KeyError, TypeError):
        print(f"Unparsable answer {choice.message.content!r} (completion {completion.id})")
        return None
    if signature not in SIGNATURE_OPTIONS:
        print(f"Unexpected answer {signature!r} (completion {completion.id})")
        return None
    return signature

def update_status(idx, document_no, new_status):
    """Set the new status in both dataframes and stamp LastUpdated with the current time,
    but only when the status actually changes. Returns True if it changed."""
    if null_status_df.at[idx, "Status"] == new_status:
        return False
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    null_status_df.loc[idx, ["Status", "LastUpdated"]] = [new_status, now]
    doc_status_df.loc[
        doc_status_df["DocumentNo"] == document_no, ["Status", "LastUpdated"]
    ] = [new_status, now]
    return True

# Send the documents that still have a null status (and a loaded PDF) one at a time
to_classify = null_status_df.index[
    null_status_df["Status"].isna() & null_status_df["PdfBytes"].notna()
]
print(f"Documents to send to {MODEL}: {len(to_classify)}")

for idx in to_classify:
    document_no = null_status_df.at[idx, "DocumentNo"]
    try:
        signature = classify_signature(null_status_df.at[idx, "PdfBytes"])
    except openai.RateLimitError as e:
        print(f"{document_no}: rate limited, retry after {e.response.headers.get('retry-after')}s")
        continue
    except openai.APIStatusError as e:
        # Show the gateway's explanation, e.g. which field of the request it rejected
        print(f"{document_no}: API error {e.status_code}: {e.response.text}")
        continue
    except openai.APIConnectionError:
        print(f"{document_no}: network error")
        continue
    if signature is None:
        continue

    if update_status(idx, document_no, signature):
        print(f"{document_no}: {signature} (LastUpdated {null_status_df.at[idx, 'LastUpdated']})")
    else:
        print(f"{document_no}: {signature} (unchanged)")

null_status_df[["KYCAssessmentID", "DocumentNo", "HasDigitalSignature", "Status", "LastUpdated"]]

/opt/conda/lib/python3.11/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Documents to send to azure-openai-gpt-5: 3
Unexpected finish reason 'None' (completion chatcmpl-EijcUWJg2Y9QAcFRr8nNatnMhJiG1)
Unexpected finish reason 'None' (completion chatcmpl-EijchFYmG9vam3bV3mryDi6LEgQwo)
Unexpected finish reason 'None' (completion chatcmpl-EijctjhCCEUn8vlZ8wvsVWhgcvWkx)


,KYCAssessmentID,DocumentNo,HasDigitalSignature,Status,LastUpdated
0,921632,001,False,None,None
1,927015,002,False,None,None
2,954350,003,False,None,None


In [10]:
import pandas as pd

EXCEL_PATH = "DocumentStatus.xlsx"

# Reload the file so every row and column in it is kept
excel_df = pd.read_excel(EXCEL_PATH, dtype=str)

# Only the documents that got a status in this run
updates = (
    null_status_df.loc[null_status_df["Status"].notna(), ["DocumentNo", "Status", "LastUpdated"]]
    .set_index("DocumentNo")
)

# Copy the new Status and LastUpdated onto the matching DocumentNo rows
mask = excel_df["DocumentNo"].isin(updates.index)
excel_df.loc[mask, "Status"] = excel_df.loc[mask, "DocumentNo"].map(updates["Status"])
excel_df.loc[mask, "LastUpdated"] = excel_df.loc[mask, "DocumentNo"].map(updates["LastUpdated"])

excel_df.to_excel(EXCEL_PATH, index=False)
print(f"Wrote {mask.sum()} updated document(s) back to {EXCEL_PATH}")
excel_df

Wrote 0 updated document(s) back to DocumentStatus.xlsx


,KYCAssessmentID,DocumentNo,CreationDate,LastUpdated,Status
0,921632,001,2026-11-10 00:00:00,NaN,NaN
1,927015,002,2026-11-10 00:00:00,NaN,NaN
2,954350,003,2026-11-10 00:00:00,NaN,NaN
